# Exercise 01: FGSM Attack

**Goal:** Implement the Fast Gradient Sign Method (FGSM) and evaluate its effectiveness at different perturbation strengths.

**Estimated time:** ~20 minutes

**Reference:** Goodfellow et al. 2014 — *Explaining and Harnessing Adversarial Examples*

## Background

FGSM is the simplest adversarial attack. Given an image $x$, true label $y$, and a model with parameters $\theta$, it computes:

$$x_\text{adv} = \text{clip}\!\left(x + \varepsilon \cdot \text{sign}(\nabla_x \mathcal{L}(\theta, x, y)),\; 0,\; 1\right)$$

The key idea: move in the direction that **increases the loss** by a small amount $\varepsilon$. The $\text{sign}()$ function normalizes the gradient so every pixel is perturbed by exactly $\varepsilon$.

**Steps:**
1. Enable gradient tracking on the input image
2. Forward pass through the model, compute cross-entropy loss
3. Backward pass to get $\nabla_x \mathcal{L}$
4. Apply perturbation: $x_\text{adv} = x + \varepsilon \cdot \text{sign}(\nabla_x \mathcal{L})$
5. Clip to valid pixel range

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import helper

torch.manual_seed(42)
np.random.seed(42)

# Load pretrained ResNet20 trained on CIFAR-10, set to eval mode
model = torch.hub.load('chenyaofo/pytorch-cifar-models', 'cifar10_resnet20', pretrained=True)
model.eval()

# Load 100 CIFAR-10 test images (32x32, no resize needed)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616])
])
testset = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False)
images, labels = next(iter(testloader))

CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

def predict(model, x):
    with torch.no_grad():
        logits = model(x)
    return logits.argmax(dim=1)

# Evaluate clean accuracy
clean_preds = predict(model, images)
print(f"Clean accuracy: {(clean_preds == labels).float().mean():.2%}")

## 🎯 TODO: Implement FGSM

Fill in the `fgsm_attack` function below.

**Hints:**
- Use `images.clone().detach().requires_grad_(True)` to create a leaf tensor with gradient tracking
- Use `nn.CrossEntropyLoss()` for the loss
- Call `.backward()` and access `.grad` on your leaf tensor
- Use `torch.clamp(x_adv, min_val, max_val)` to clip the result

In [ ]:
def fgsm_attack(model, images, labels, epsilon):
    """
    Implement FGSM attack.
    Args:
        model: pretrained classifier
        images: batch of images (N, C, H, W), requires_grad will be set
        labels: true labels (N,)
        epsilon: perturbation magnitude
    Returns:
        adversarial images (N, C, H, W), clipped to valid range
    """
    # 🎯 TODO: Enable gradient tracking on images
    # HINT: use images.clone().detach().requires_grad_(True) to create a leaf tensor
    x = None

    # 🎯 TODO: Forward pass + compute cross-entropy loss
    loss = None

    # 🎯 TODO: Backward pass to get gradient w.r.t. images
    # loss.backward()

    # 🎯 TODO: Apply perturbation: x_adv = x + epsilon * sign(grad)
    x_adv = None

    # 🎯 TODO: Clip to valid range (use image min/max for normalized images)
    return x_adv

## 🎯 Evaluate FGSM at Multiple Epsilon Values

Test your FGSM implementation at `epsilon` values: `[0.01, 0.05, 0.1, 0.3, 0.5]`.

For each epsilon:
1. Generate adversarial images
2. Compute adversarial accuracy
3. Print the result

In [ ]:
epsilons = [0.01, 0.05, 0.1, 0.3, 0.5]
adv_accuracies = []

print(f"{'Epsilon':>10}  {'Adv Accuracy':>14}")
print("-" * 28)
for eps in epsilons:
    adv_images = fgsm_attack(model, images, labels, eps)
    adv_preds = predict(model, adv_images)
    acc = (adv_preds == labels).float().mean().item()
    adv_accuracies.append(acc)
    print(f"{eps:>10.3f}  {acc:>14.2%}")

## Plot Accuracy vs. Epsilon

Plot clean accuracy (horizontal dashed line) and adversarial accuracy vs. epsilon.

In [ ]:
clean_acc = (clean_preds == labels).float().mean().item()
helper.plot_accuracy_curves(
    x_values=epsilons,
    y_dict={"Adversarial accuracy": adv_accuracies},
    xlabel="ε (perturbation strength)",
    title="FGSM: Accuracy vs. Perturbation Strength (ResNet20, CIFAR-10)",
    hlines={f"Clean accuracy ({clean_acc:.2%})": clean_acc},
)

## Visualize Adversarial Examples

For $\varepsilon = 0.1$, show 5 examples in a 3-row grid:
- **Row 1:** Original image + true label
- **Row 2:** Perturbation (scaled for visibility)
- **Row 3:** Adversarial image + predicted label (red = attack succeeded)

In [ ]:
eps_viz = 0.1
n = 5
adv_images_viz = fgsm_attack(model, images, labels, eps_viz)
adv_preds_viz  = predict(model, adv_images_viz)

# Normalize perturbation to [0,1] for display
pert_images = []
for i in range(n):
    pert = adv_images_viz[i] - images[i]
    pert_images.append((pert - pert.min()) / (pert.max() - pert.min() + 1e-8))

helper.plot_image_grid(
    rows=[
        {
            "images": [images[i] for i in range(n)],
            "titles": [CIFAR10_CLASSES[labels[i].item()] for i in range(n)],
            "ylabel": "Original",
        },
        {
            "images": pert_images,
            "titles": ["Noise (scaled)" for _ in range(n)],
            "denorm": False,
            "ylabel": "Perturbation",
        },
        {
            "images": [adv_images_viz[i] for i in range(n)],
            "titles": [CIFAR10_CLASSES[adv_preds_viz[i].item()] for i in range(n)],
            "colors": ["red" if adv_preds_viz[i] != labels[i] else "green" for i in range(n)],
            "ylabel": "Adversarial",
        },
    ],
    suptitle=f"FGSM Attack (ε={eps_viz}) — Red title = attack succeeded",
)